In [1]:
import gc
import sqlite3
import time

import torch
from pylate import indexes, models
from tqdm.auto import tqdm

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [3]:
# Load the ColBERT model
model = models.ColBERT(
    model_name_or_path="lightonai/colbertv2.0",
    device = device
)

In [4]:
OVERRIDE_INDEX = True  # set to False to load an existing index

index = indexes.Voyager(
    index_folder="../data/processed/pylate_index",
    index_name="esci_data_index",
    override=OVERRIDE_INDEX,
)

In [5]:
LIMIT = 20000 # set to None to index all documents

conn = sqlite3.connect("../data/processed/products_dataset.db")
cursor = conn.cursor()

query = "SELECT pid, product_text FROM products"
if LIMIT:
    query += f" LIMIT {LIMIT}"
cursor.execute(query)

In [6]:
ENCODE_BATCH_SIZE = 128    # docs per model.encode() call (GPU memory bound)
INDEX_BATCH_SIZE = 10_000  # docs accumulated before calling add_documents (RAM bound)

def iter_batches(cursor, batch_size):
    while chunk := cursor.fetchmany(batch_size):
        ids, texts = zip(*chunk)
        yield list(ids), list(texts)

def flush_to_index(ids, embeddings):
    index.add_documents(documents_ids=ids, documents_embeddings=embeddings)
    ids.clear()
    embeddings.clear()

total_start = time.time()
encoding_time = 0
indexing_time = 0
pending_ids = []
pending_embeddings = []

n_batches = (LIMIT // ENCODE_BATCH_SIZE) if LIMIT else None
for batch_ids, batch_texts in tqdm(iter_batches(cursor, ENCODE_BATCH_SIZE), total=n_batches, unit="batch", desc="Encoding"):
    t0 = time.time()
    embeddings = model.encode(
        batch_texts,
        is_query=False,
        batch_size=ENCODE_BATCH_SIZE,
        pool_factor=2,
        show_progress_bar=False,
    )
    if isinstance(embeddings, list):
        embeddings = [e.cpu() if hasattr(e, "cpu") else e for e in embeddings]
    encoding_time += time.time() - t0

    pending_ids.extend(batch_ids)
    pending_embeddings.extend(embeddings)

    if device == "mps":
        torch.mps.empty_cache()

    if len(pending_ids) >= INDEX_BATCH_SIZE:
        print(f"Flushing {len(pending_ids)} documents to index...")
        t0 = time.time()
        flush_to_index(pending_ids, pending_embeddings)
        indexing_time += time.time() - t0

if pending_ids:
    print(f"Flushing final {len(pending_ids)} documents to index...")
    t0 = time.time()
    flush_to_index(pending_ids, pending_embeddings)
    indexing_time += time.time() - t0

conn.close()

if device == "mps":
    torch.mps.empty_cache()
gc.collect()

print(f"Encoding time: {encoding_time:.1f}s")
print(f"Indexing time: {indexing_time:.1f}s")
print(f"Total time:    {time.time() - total_start:.1f}s")

Encoding:   0%|          | 0/156 [00:00<?, ?batch/s]

Flushing 10112 documents to index...


Adding documents to the index (bs=2000): 100%|██████████| 6/6 [00:29<00:00,  4.99s/it]


Flushing final 9888 documents to index...


Adding documents to the index (bs=2000): 100%|██████████| 5/5 [00:29<00:00,  5.82s/it]


Encoding time: 573.5s
Indexing time: 62.0s
Total time:    639.2s
